# Covariate Balance Diagnostics Benchmark

This notebook demonstrates that `causers.balance_check` produces correct balance
statistics and benchmarks its performance on large datasets.

In [1]:
import sys
import time

import numpy as np
import polars as pl

import causers

print(f"causers version: {causers.__version__}")
print(f"numpy version:   {np.__version__}")
print(f"polars version:  {pl.__version__}")
print(f"Python {sys.version}")

causers version: 0.8.0
numpy version:   2.4.2
polars version:  1.37.1
Python 3.12.10 (main, Apr  9 2025, 03:49:38) [Clang 20.1.0 ]


In [2]:
SEED = 42


def time_function(func, *args, n_iter=10, warmup=2, **kwargs):
    """Benchmark a function and return median/min/max ms."""
    for _ in range(warmup):
        func(*args, **kwargs)
    times = []
    result = None
    for _ in range(n_iter):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = (time.perf_counter() - start) * 1000
        times.append(elapsed)
    return {
        "result": result,
        "median_ms": np.median(times),
        "min_ms": min(times),
        "max_ms": max(times),
    }


def generate_balance_data(n_obs, n_covariates, weighted=False, seed=SEED):
    """Generate a DataFrame with binary treatment and continuous covariates."""
    rng = np.random.default_rng(seed)
    treatment = rng.integers(0, 2, size=n_obs)
    data = {"treatment": treatment}
    for i in range(n_covariates):
        # Introduce a slight shift for treated group so balance isn't perfect
        shift = 0.1 * (i + 1) / n_covariates
        data[f"x{i}"] = rng.standard_normal(n_obs) + treatment * shift
    if weighted:
        data["weights"] = rng.uniform(0.1, 3.0, size=n_obs)
    return pl.DataFrame(data)


print("Helper functions defined.")

Helper functions defined.


## Parity Test

Compare `causers.balance_check` against a manual NumPy calculation on a small
hand-constructed dataset to verify correctness of SMD and variance ratio.

In [3]:
print("=" * 60)
print("PARITY TEST: SMD and Variance Ratio")
print("=" * 60)

# Small hand-constructed dataset
df_small = pl.DataFrame({
    "treated": [1, 1, 1, 0, 0, 0, 0, 0],
    "age":     [30.0, 35.0, 40.0, 50.0, 55.0, 45.0, 48.0, 52.0],
    "income":  [60000.0, 65000.0, 70000.0, 45000.0, 50000.0, 48000.0, 47000.0, 51000.0],
})

result = causers.balance_check(df_small, "treated", ["age", "income"])

# Manual NumPy calculation
treated_mask = np.array([True, True, True, False, False, False, False, False])
age = np.array([30.0, 35.0, 40.0, 50.0, 55.0, 45.0, 48.0, 52.0])
income = np.array([60000.0, 65000.0, 70000.0, 45000.0, 50000.0, 48000.0, 47000.0, 51000.0])

for col_name, values in [("age", age), ("income", income)]:
    t_vals = values[treated_mask]
    c_vals = values[~treated_mask]
    mean_t, mean_c = t_vals.mean(), c_vals.mean()
    var_t = t_vals.var(ddof=1)
    var_c = c_vals.var(ddof=1)
    pooled_sd = np.sqrt((var_t + var_c) / 2)
    manual_smd = (mean_t - mean_c) / pooled_sd
    manual_vr = var_t / var_c

    causers_smd = result.smd[col_name]
    causers_vr = result.variance_ratio[col_name]

    smd_match = abs(causers_smd - manual_smd) < 1e-6
    vr_match = abs(causers_vr - manual_vr) < 1e-6

    print(f"\n  {col_name}:")
    print(f"    SMD  — causers: {causers_smd:.6f}, manual: {manual_smd:.6f}  [{'PASS' if smd_match else 'FAIL'}]")
    print(f"    VR   — causers: {causers_vr:.6f}, manual: {manual_vr:.6f}  [{'PASS' if vr_match else 'FAIL'}]")

PARITY TEST: SMD and Variance Ratio

  age:
    SMD  — causers: -3.375264, manual: -3.375264  [PASS]
    VR   — causers: 1.724138, manual: 1.724138  [PASS]

  income:
    SMD  — causers: 4.288003, manual: 4.288003  [PASS]
    VR   — causers: 4.385965, manual: 4.385965  [PASS]


/var/folders/k0/m1_drbkj7r53yw9cpj1kvwvm0000gn/T/ipykernel_114/1680490632.py:12: UserWarning: Large imbalance detected for covariate 'age': SMD = -3.3753
  result = causers.balance_check(df_small, "treated", ["age", "income"])
/var/folders/k0/m1_drbkj7r53yw9cpj1kvwvm0000gn/T/ipykernel_114/1680490632.py:12: UserWarning: Large imbalance detected for covariate 'income': SMD = 4.2880
  result = causers.balance_check(df_small, "treated", ["age", "income"])
/var/folders/k0/m1_drbkj7r53yw9cpj1kvwvm0000gn/T/ipykernel_114/1680490632.py:12: UserWarning: Extreme variance ratio for covariate 'income': 4.3860
  result = causers.balance_check(df_small, "treated", ["age", "income"])
/var/folders/k0/m1_drbkj7r53yw9cpj1kvwvm0000gn/T/ipykernel_114/1680490632.py:12: UserWarning: Small treatment group: n_treated = 3. Balance statistics may be unreliable.
  result = causers.balance_check(df_small, "treated", ["age", "income"])
/var/folders/k0/m1_drbkj7r53yw9cpj1kvwvm0000gn/T/ipykernel_114/1680490632.py:12:

## Result Exploration

Demonstrate the convenience methods and direct attribute access on `BalanceCheckResult`.

In [4]:
print("=" * 60)
print("RESULT EXPLORATION")
print("=" * 60)

print("\n--- result.summary() ---")
print(result.summary())

print("\n--- result.imbalanced(threshold=0.1) ---")
print(result.imbalanced(threshold=0.1))

print("\n--- result.to_dataframe() ---")
print(result.to_dataframe())

print("\n--- Direct attribute access ---")
print(f"  n_treated:  {result.n_treated}")
print(f"  n_control:  {result.n_control}")
print(f"  covariates: {result.covariates}")
print(f"  is_weighted: {result.is_weighted}")
print(f"  smd:        {result.smd}")

RESULT EXPLORATION

--- result.summary() ---
shape: (2, 7)
┌───────────┬──────────────┬──────────────┬────────────┬─────────────┬───────────┬────────────────┐
│ covariate ┆ mean_treated ┆ mean_control ┆ sd_treated ┆ sd_control  ┆ smd       ┆ variance_ratio │
│ ---       ┆ ---          ┆ ---          ┆ ---        ┆ ---         ┆ ---       ┆ ---            │
│ str       ┆ f64          ┆ f64          ┆ f64        ┆ f64         ┆ f64       ┆ f64            │
╞═══════════╪══════════════╪══════════════╪════════════╪═════════════╪═══════════╪════════════════╡
│ age       ┆ 35.0         ┆ 50.0         ┆ 5.0        ┆ 3.807887    ┆ -3.375264 ┆ 1.724138       │
│ income    ┆ 65000.0      ┆ 48200.0      ┆ 5000.0     ┆ 2387.467277 ┆ 4.288003  ┆ 4.385965       │
└───────────┴──────────────┴──────────────┴────────────┴─────────────┴───────────┴────────────────┘

--- result.imbalanced(threshold=0.1) ---
['age', 'income']

--- result.to_dataframe() ---
shape: (2, 9)
┌───────────┬───────────┬───────────

## Timing Benchmarks

Measure performance across different dataset sizes and covariate counts.

In [5]:
BALANCE_CONFIGS = [
    (100_000, 50, False, "100K obs / 50 covs"),
    (1_000_000, 100, False, "1M obs / 100 covs"),
    (100_000, 50, True, "100K obs / 50 covs (weighted)"),
]

print("=" * 80)
print("TIMING BENCHMARKS")
print("=" * 80)

bench_results = []
for n_obs, n_covs, weighted, label in BALANCE_CONFIGS:
    print(f"  {label}...", end=" ", flush=True)
    df_bench = generate_balance_data(n_obs, n_covs, weighted=weighted)
    cov_cols = [f"x{i}" for i in range(n_covs)]
    w = "weights" if weighted else None

    def run(_df=df_bench, _cols=cov_cols, _w=w):
        return causers.balance_check(_df, "treatment", _cols, weights=_w)

    timing = time_function(run)
    bench_results.append({"label": label, "median_ms": timing["median_ms"],
                          "min_ms": timing["min_ms"], "max_ms": timing["max_ms"]})
    print(f"{timing['median_ms']:.2f}ms (min={timing['min_ms']:.2f}, max={timing['max_ms']:.2f})")

print("\nBenchmark complete!")

TIMING BENCHMARKS
  100K obs / 50 covs... 

37.12ms (min=36.40, max=46.67)
  1M obs / 100 covs... 

680.26ms (min=658.04, max=851.08)
  100K obs / 50 covs (weighted)... 

52.64ms (min=50.25, max=55.04)

Benchmark complete!


## Weighted Analysis Demo

Generate IPW-style weights and run a weighted balance check to demonstrate
effective sample size (ESS) reporting.

In [6]:
print("=" * 60)
print("WEIGHTED ANALYSIS")
print("=" * 60)

rng = np.random.default_rng(SEED)
n = 1000
treatment = rng.integers(0, 2, size=n)
x1 = rng.standard_normal(n) + treatment * 0.5
x2 = rng.standard_normal(n) + treatment * 0.3

# Simulate IPW weights (larger weights for underrepresented propensity strata)
propensity = 1 / (1 + np.exp(-(0.5 * x1 + 0.3 * x2)))
ipw = np.where(treatment == 1, 1 / propensity, 1 / (1 - propensity))

df_weighted = pl.DataFrame({
    "treatment": treatment,
    "x1": x1,
    "x2": x2,
    "ipw": ipw,
})

# Unweighted
res_uw = causers.balance_check(df_weighted, "treatment", ["x1", "x2"])
print("\nUnweighted:")
print(f"  SMD x1: {res_uw.smd['x1']:.4f}")
print(f"  SMD x2: {res_uw.smd['x2']:.4f}")
print(f"  ESS treated: {res_uw.ess_treated}")
print(f"  ESS control: {res_uw.ess_control}")

# Weighted
res_w = causers.balance_check(df_weighted, "treatment", ["x1", "x2"], weights="ipw")
print("\nWeighted (IPW):")
print(f"  SMD x1: {res_w.smd['x1']:.4f}")
print(f"  SMD x2: {res_w.smd['x2']:.4f}")
print(f"  ESS treated: {res_w.ess_treated:.1f}")
print(f"  ESS control: {res_w.ess_control:.1f}")
print(f"  is_weighted: {res_w.is_weighted}")

WEIGHTED ANALYSIS

Unweighted:
  SMD x1: 0.6000
  SMD x2: 0.2323
  ESS treated: None
  ESS control: None

Weighted (IPW):
  SMD x1: 0.1050
  SMD x2: -0.0564
  ESS treated: 469.5
  ESS control: 435.8
  is_weighted: True


/var/folders/k0/m1_drbkj7r53yw9cpj1kvwvm0000gn/T/ipykernel_114/3816800636.py:23: UserWarning: Large imbalance detected for covariate 'x1': SMD = 0.6000
  res_uw = causers.balance_check(df_weighted, "treatment", ["x1", "x2"])


## Categorical Covariate Demo

String (categorical) columns are automatically one-hot expanded. The result
covariates list shows the expanded indicator names.

In [7]:
print("=" * 60)
print("CATEGORICAL COVARIATE EXPANSION")
print("=" * 60)

df_cat = pl.DataFrame({
    "treatment": [1, 1, 1, 1, 0, 0, 0, 0],
    "age":       [25.0, 30.0, 35.0, 28.0, 40.0, 45.0, 38.0, 42.0],
    "region":    ["north", "south", "north", "east", "south", "east", "north", "south"],
})

res_cat = causers.balance_check(df_cat, "treatment", ["age", "region"])

print(f"\nInput covariates:  ['age', 'region']")
print(f"Output covariates: {res_cat.covariates}")
print()
print(res_cat.summary())

CATEGORICAL COVARIATE EXPANSION

Input covariates:  ['age', 'region']
Output covariates: ['age', 'region_east', 'region_north', 'region_south']

shape: (4, 7)
┌──────────────┬──────────────┬──────────────┬────────────┬────────────┬──────────┬────────────────┐
│ covariate    ┆ mean_treated ┆ mean_control ┆ sd_treated ┆ sd_control ┆ smd      ┆ variance_ratio │
│ ---          ┆ ---          ┆ ---          ┆ ---        ┆ ---        ┆ ---      ┆ ---            │
│ str          ┆ f64          ┆ f64          ┆ f64        ┆ f64        ┆ f64      ┆ f64            │
╞══════════════╪══════════════╪══════════════╪════════════╪════════════╪══════════╪════════════════╡
│ age          ┆ 29.5         ┆ 41.25        ┆ 4.203173   ┆ 2.986079   ┆ -3.22291 ┆ 1.981308       │
│ region_east  ┆ 0.25         ┆ 0.25         ┆ 0.5        ┆ 0.5        ┆ 0.0      ┆ 1.0            │
│ region_north ┆ 0.5          ┆ 0.25         ┆ 0.57735    ┆ 0.5        ┆ 0.46291  ┆ 1.333333       │
│ region_south ┆ 0.25         ┆ 0

/var/folders/k0/m1_drbkj7r53yw9cpj1kvwvm0000gn/T/ipykernel_114/4107338143.py:11: UserWarning: Large imbalance detected for covariate 'age': SMD = -3.2229
  res_cat = causers.balance_check(df_cat, "treatment", ["age", "region"])
/var/folders/k0/m1_drbkj7r53yw9cpj1kvwvm0000gn/T/ipykernel_114/4107338143.py:11: UserWarning: Large imbalance detected for covariate 'region_north': SMD = 0.4629
  res_cat = causers.balance_check(df_cat, "treatment", ["age", "region"])
/var/folders/k0/m1_drbkj7r53yw9cpj1kvwvm0000gn/T/ipykernel_114/4107338143.py:11: UserWarning: Large imbalance detected for covariate 'region_south': SMD = -0.4629
  res_cat = causers.balance_check(df_cat, "treatment", ["age", "region"])
/var/folders/k0/m1_drbkj7r53yw9cpj1kvwvm0000gn/T/ipykernel_114/4107338143.py:11: UserWarning: Small treatment group: n_treated = 4. Balance statistics may be unreliable.
  res_cat = causers.balance_check(df_cat, "treatment", ["age", "region"])
/var/folders/k0/m1_drbkj7r53yw9cpj1kvwvm0000gn/T/ipyker

## Summary

In [8]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"\n{'Config':<40} {'Median (ms)':>12} {'Min (ms)':>10} {'Max (ms)':>10}")
print("-" * 74)
for r in bench_results:
    print(f"{r['label']:<40} {r['median_ms']:>12.2f} {r['min_ms']:>10.2f} {r['max_ms']:>10.2f}")

print("\nBenchmark complete!")

SUMMARY

Config                                    Median (ms)   Min (ms)   Max (ms)
--------------------------------------------------------------------------
100K obs / 50 covs                              37.12      36.40      46.67
1M obs / 100 covs                              680.26     658.04     851.08
100K obs / 50 covs (weighted)                   52.64      50.25      55.04

Benchmark complete!
